In [1]:
import pandas as pd

In [2]:
movies_df = pd.read_csv("../datasets/imdb_top_rated.csv")
movies_df.head(16)

,web-scraper-order,web-scraper-start-url,Movie title,Rating,Movie links,Movie links-href,Director,Writers,Stars
0,1771933098-1,https://www.imdb.com/chart/top/,The Shawshank Redemption,9.3,NaN,NaN,NaN,NaN,NaN
1,1771933098-2,https://www.imdb.com/chart/top/,The Godfather,9.2,NaN,NaN,NaN,NaN,NaN
2,1771933098-3,https://www.imdb.com/chart/top/,The Dark Knight,9.1,NaN,NaN,NaN,NaN,NaN
3,1771933098-4,https://www.imdb.com/chart/top/,The Godfather Part II,9.0,NaN,NaN,NaN,NaN,NaN
4,1771933098-5,https://www.imdb.com/chart/top/,12 Angry Men,9.0,NaN,NaN,NaN,NaN,NaN
5,1771933098-6,https://www.imdb.com/chart/top/,The Lord of the Rings: The Return of the King,9.0,NaN,NaN,NaN,NaN,NaN
6,1771933098-7,https://www.imdb.com/chart/top/,Schindler's List,9.0,NaN,NaN,NaN,NaN,NaN
7,1771933098-8,https://www.imdb.com/chart/top/,The Lord of the Rings: The Fellowship of the Ring,8.9,NaN,NaN,NaN,NaN,NaN
8,1771933098-9,https://www.imdb.com/chart/top/,Pulp Fiction,8.8,NaN,NaN,NaN,NaN,NaN
9,1771933098-10,https://www.imdb.com/chart/top/,"The Good, the Bad and the Ugly",8.8,NaN,NaN,NaN,NaN,NaN


I couldn't get webscraper.io to scrape the data in the correct format. As a result it hasn't been concentated together correctly. In order to fix this I'll try to do some data manipulation on the dataset.

## Dropping columns

In [3]:
# Drop useless columns with .drop()
movies_df.drop(['web-scraper-order', 'web-scraper-start-url', 'Movie links-href'], axis=1, inplace=True)
movies_df.tail(6)

,Movie title,Rating,Movie links,Director,Writers,Stars
2000,NaN,NaN,The Shawshank Redemption,Frank Darabont,NaN,NaN
2001,NaN,NaN,The Shawshank Redemption,NaN,Stephen King,NaN
2002,NaN,NaN,The Shawshank Redemption,NaN,Frank Darabont,NaN
2003,NaN,NaN,The Shawshank Redemption,NaN,NaN,Tim Robbins
2004,NaN,NaN,The Shawshank Redemption,NaN,NaN,Morgan Freeman
2005,NaN,NaN,The Shawshank Redemption,NaN,NaN,Bob Gunton


## Aggregate function
A function that combines a list of values (e.g. a specific row or column) into a single value using a function. A simple example is using `sum()` to sum all the values in a column.

In [4]:
# Join the Directors, Writers, Stars together into one row per unique Movie link 
grouped_movies_df = movies_df.groupby('Movie links').agg({
    'Director': lambda x: ', '.join(set(x.dropna())),
    'Writers': lambda x: ', '.join(set(x.dropna())),
    'Stars': lambda x: ', '.join(set(x.dropna()))
})

# Reset the index so that the "Movie links" data won't be used as the index
grouped_movies_df.reset_index(inplace=True)

# Check if the groupby() operation has been successful
# movies2_df[movies2_df['Movie links'] == '16. The Matrix']
# movies2_df.columns.tolist()
# movies2_df.head()
grouped_movies_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Movie links  250 non-null    object
 1   Director     250 non-null    object
 2   Writers      250 non-null    object
 3   Stars        250 non-null    object
dtypes: object(4)
memory usage: 7.9+ KB


In [5]:
grouped_movies_df.rename(columns={'Movie links': 'Movie title'}, inplace=True)
grouped_movies_df

,Movie title,Director,Writers,Stars
0,12 Angry Men,Sidney Lumet,Reginald Rose,"Henry Fonda, Martin Balsam, Lee J. Cobb"
1,12 Years a Slave,Steve McQueen,"John Ridley, Solomon Northup","Michael Fassbender, Michael Kenneth Williams, ..."
2,12th Fail,Vidhu Vinod Chopra,"Anurag Pathak, Jaskunwar Kohli, Vidhu Vinod Ch...","Medha Shankr, Jaskunwar Kohli, Vidhu Vinod Cho..."
3,1917,Sam Mendes,"Sam Mendes, Krysty Wilson-Cairns","Daniel Mays, Dean-Charles Chapman, George MacKay"
4,2001: A Space Odyssey,Stanley Kubrick,"Stanley Kubrick, Arthur C. Clarke","Keir Dullea, Gary Lockwood, William Sylvester"
...,...,...,...,...
245,Wild Strawberries,Ingmar Bergman,Ingmar Bergman,"Ingrid Thulin, Bibi Andersson, Victor Sjöström"
246,Wild Tales,Damián Szifron,"Damián Szifron, Germán Servidio, Julian Loyola","María Marull, Mónica Villa, Darío Grandinetti"
247,Witness for the Prosecution,Billy Wilder,"Harry Kurnitz, Billy Wilder, Agatha Christie","Tyrone Power, Marlene Dietrich, Billy Wilder, ..."
248,Yojimbo,Akira Kurosawa,"Akira Kurosawa, Ryûzô Kikushima","Tatsuya Nakadai, Eijirô Tôno, Toshirô Mifune"


## Merging

In [6]:
# Using merge() creates a new DataFrame that doesn't fill the NaN values correctly
#movies_df_all = pd.merge(movies_df, movies2_df, left_on = 'Movie title', right_on = 'Movie links')

# fillna() however does exactly what I want, but is index dependent so it doesn't work
# movies_df.fillna(movies2_df, inplace=True)

# I also tried combine_first as suggested in a forum post, but didn't work either because
# its also index dependent
# movies2_df.combine_first(movies_df)

# I ended up just creating two DataFrames from the original and merging those:

# Create a 'subset' and drop all rows with NaN values
movies_rating_df = movies_df[['Movie title', 'Rating']].dropna()
movies_rating_df

# Merge the two together on "Movie title"
movies_df_all = pd.merge(movies_rating_df, grouped_movies_df, on='Movie title')
movies_df_all

,Movie title,Rating,Director,Writers,Stars
0,The Shawshank Redemption,9.3,Frank Darabont,"Stephen King, Frank Darabont","Tim Robbins, Bob Gunton, Morgan Freeman"
1,The Godfather,9.2,Francis Ford Coppola,"Francis Ford Coppola, Mario Puzo","Marlon Brando, Al Pacino, James Caan"
2,The Dark Knight,9.1,Christopher Nolan,"David S. Goyer, Jonathan Nolan, Christopher Nolan","Jonathan Nolan, Heath Ledger, David S. Goyer, ..."
3,The Godfather Part II,9.0,Francis Ford Coppola,"Francis Ford Coppola, Mario Puzo","Al Pacino, Robert Duvall, Robert De Niro"
4,12 Angry Men,9.0,Sidney Lumet,Reginald Rose,"Henry Fonda, Martin Balsam, Lee J. Cobb"
...,...,...,...,...,...
245,To Be or Not to Be,8.1,Ernst Lubitsch,"Ernst Lubitsch, Edwin Justus Mayer, Melchior L...","Robert Stack, Jack Benny, Carole Lombard"
246,Gangs of Wasseypur,8.2,Anurag Kashyap,"Akhilesh Jaiswal, Sachin K. Ladia, Anurag Kashyap","Nawazuddin Siddiqui, Manoj Bajpayee, Akhilesh ..."
247,Drishyam,8.2,Nishikant Kamat,"Upendra Sidhaye, Jeethu Joseph","Tabu, Ajay Devgn, Shriya Saran"
248,The Grapes of Wrath,8.1,John Ford,"Nunnally Johnson, John Steinbeck","John Carradine, Henry Fonda, Jane Darwell"


## Tests
Some tests generated by an LLM to better understand what the groupby() and agg() functions do

In [7]:
import pandas as pd

# Example DataFrame
data = {
    'A': [1, 1, 1],
    'B': ['A', 'B', 'C'],
    'C': [10, 20, 30],
    'D': [40, 50, 60]
}
df = pd.DataFrame(data)
df

,A,B,C,D
0,1,A,10,40
1,1,B,20,50
2,1,C,30,60


In [8]:
# Group by column 'A' and apply the same aggregation function to multiple columns
result = df.groupby('A').agg({
    'B': ' '.join,
    'C': 'sum',
    'D': 'sum'
}).reset_index()

result


,A,B,C,D
0,1,A B C,60,150
